In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.goldfunctions import date_key, write_gold, build_snapshot, check_no_duplicates

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("s2g_factshiftsnapshot")

# ============================================================
# 2. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

logger.info("Reading Gold configuration from %s", config_path)

# ============================================================
# 3. READ CONFIG
# ============================================================
with open(config_path, "r") as file:
    config = json.load(file)

# ============================================================
# 4. CONFIG SECTIONS
# ============================================================
table_config = config["fact_shift_snapshot"]

source_table   = table_config["source_table"]
snapshot_table = table_config["snapshot_table"]
target_table   = table_config["target_table"]
key_column     = table_config["key_column"]
start_column   = table_config["start_column"]
end_column     = table_config["end_column"]
hours_column   = table_config["hours_column"]

# ============================================================
# 5. READ SILVER FACT AND SNAPSHOT DATES
# ============================================================
logger.info("Reading Silver table: %s", source_table)

fact_df     = spark.table(source_table)
snapshot_df = spark.table(snapshot_table)

# ============================================================
# 6. BUILD SNAPSHOTS
# ============================================================
df = build_snapshot(fact_df, snapshot_df)

# ============================================================
# 7. ADD KEYS AND MEASURES
# ============================================================
df = (
    df.withColumn("date_key", date_key(start_column))
      .withColumn(
          hours_column,
          (F.unix_timestamp(end_column) - F.unix_timestamp(start_column)) / 3600
      )
)

# ============================================================
# 8. VALIDATE
# ============================================================
check_no_duplicates(df, key_column)

# ============================================================
# 9. DISPLAY DATA
# ============================================================
display(df)

# ============================================================
# 10. WRITE GOLD TABLE
# ============================================================
write_gold(spark, df, target_table)

# ============================================================
# 11. COMPLETED
# ============================================================
logger.info("Gold fact_shift_snapshot processing completed")